In [1]:
import optuna
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split,GridSearchCV,cross_validate
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv('balanced.csv')
col =['Sex','MaritalStatus']
ct = ColumnTransformer([('ohe',OneHotEncoder(),col)],remainder='passthrough')
x = df.drop('FraudFound_P',axis=1)
y = df['FraudFound_P']
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,shuffle=True,stratify=y)
x_train = ct.fit_transform(x_train)
x_test = ct.transform(x_test)

In [3]:
import optuna

def objective(trial):
    C = trial.suggest_float('C',0.02,5)
    gamma = trial.suggest_float('gamma',25,35)
    svc = SVC(C=C,gamma=gamma,probability=True,kernel='linear')
    svc.fit(x_train,y_train)
    y_pred = svc.predict(x_test)
    return f1_score(y_test,y_pred)


study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(objective,n_trials=100)
print(study.best_params)
print(study.best_value)

[I 2026-05-08 22:14:20,277] A new study created in memory with name: no-name-afe8bd82-6abc-499d-83da-97c63c9a917d
[I 2026-05-08 22:14:22,044] Trial 0 finished with value: 0.7483443708609272 and parameters: {'C': 4.80539176495672, 'gamma': 26.53402617410623}. Best is trial 0 with value: 0.7483443708609272.
[I 2026-05-08 22:14:23,650] Trial 1 finished with value: 0.7483443708609272 and parameters: {'C': 4.602248276668851, 'gamma': 27.119886623783025}. Best is trial 0 with value: 0.7483443708609272.
[I 2026-05-08 22:14:24,192] Trial 2 finished with value: 0.7483443708609272 and parameters: {'C': 1.4473487892444166, 'gamma': 30.5358276473684}. Best is trial 0 with value: 0.7483443708609272.
[I 2026-05-08 22:14:25,261] Trial 3 finished with value: 0.7483443708609272 and parameters: {'C': 3.1015927511409864, 'gamma': 33.93757854717204}. Best is trial 0 with value: 0.7483443708609272.
[I 2026-05-08 22:14:26,803] Trial 4 finished with value: 0.7483443708609272 and parameters: {'C': 4.722494390

{'C': 0.020671116106580872, 'gamma': 31.273113706877883}
0.7701492537313432


In [4]:
def objective(trial):
    par = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'use_label_encoder': False,
        'eval_metric': 'logloss'
    }
    
    model = XGBClassifier(**par)
    
    score = cross_validate(model, x_train, y_train, cv=5, n_jobs=-1, scoring='f1')
    return score['test_score'].mean()


In [5]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=200)
print(study.best_params)
print(study.best_value)

[I 2026-05-08 22:15:07,878] A new study created in memory with name: no-name-0ce967ba-7479-4608-80b9-0600add56789
[I 2026-05-08 22:15:14,974] Trial 0 finished with value: 0.7264064737487738 and parameters: {'n_estimators': 269, 'max_depth': 3, 'learning_rate': 0.015560585410736766, 'subsample': 0.9331523061543351, 'colsample_bytree': 0.6239981840552208, 'gamma': 3.308239264436793}. Best is trial 0 with value: 0.7264064737487738.
[I 2026-05-08 22:15:18,741] Trial 1 finished with value: 0.7038857989567879 and parameters: {'n_estimators': 798, 'max_depth': 9, 'learning_rate': 0.09355312257532139, 'subsample': 0.569098273167814, 'colsample_bytree': 0.6745084385478262, 'gamma': 2.4356759182875107}. Best is trial 0 with value: 0.7264064737487738.
[I 2026-05-08 22:15:18,960] Trial 2 finished with value: 0.6778007715212796 and parameters: {'n_estimators': 689, 'max_depth': 4, 'learning_rate': 0.05980775997266479, 'subsample': 0.7185710020865974, 'colsample_bytree': 0.7762891476586761, 'gamma':

{'n_estimators': 406, 'max_depth': 7, 'learning_rate': 0.11969221233309466, 'subsample': 0.8281744829910521, 'colsample_bytree': 0.522545307309381, 'gamma': 4.813060322698257}
0.7384662334416381


In [6]:
# svm best

svc = SVC(C= 0.02044417140421828, gamma= 25.088998615989063,probability=True,kernel='linear')
svc.fit(x_train,y_train)
y_pred = svc.predict(x_test)
print(f1_score(y_test,y_pred))

0.7701492537313432


In [7]:
print(accuracy_score(y_test,y_pred))
print(precision_score(y_test,y_pred))
print(recall_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

0.7269503546099291
0.6649484536082474
0.9148936170212766
[[ 76  65]
 [ 12 129]]


In [8]:
import joblib
joblib.dump(svc,'svc_model.pkl')

['svc_model.pkl']

In [10]:
joblib.dump(ct,'ct.pkl')

['ct.pkl']